# 📧 Phân Tích Khám Phá Dữ Liệu (EDA) & Tiền Xử Lý Dữ Liệu
# Dataset: Phishing Email Detection

---

## Thông tin Dataset

| Thuộc tính | Giá trị |
|---|---|
| **Tên dataset** | Phishing Email Detection Dataset |
| **Mô tả** | Tập dữ liệu chứa nội dung email (Email Text) đã được gán nhãn là email an toàn (Safe Email) hoặc email lừa đảo (Phishing Email), phục vụ bài toán phân loại nhị phân trong lĩnh vực An ninh mạng (Cybersecurity). |
| **Định dạng** | CSV |
| **Cột mục tiêu (Target)** | `Email Type` — Gồm 2 giá trị: `Safe Email` (0) và `Phishing Email` (1) |
| **Phương pháp luận** | CRISP-DM (Cross-Industry Standard Process for Data Mining) |

## Dataset Information (English)

| Attribute | Value |
|---|---|
| **Dataset Name** | Phishing Email Detection Dataset |
| **Description** | A labeled email corpus containing email body text classified as either Safe Email or Phishing Email, designed for binary text classification in the Cybersecurity domain. |
| **Format** | CSV |
| **Target Variable** | `Email Type` — 2 classes: `Safe Email` (0) and `Phishing Email` (1) |
| **Methodology** | CRISP-DM (Cross-Industry Standard Process for Data Mining) |

---

## Mục lục

1. [Section 1: Khởi tạo & Thống kê Mô tả Tổng quan](#section-1)
2. [Section 2: Phân Tích Đơn Biến (Univariate Analysis)](#section-2)
3. [Section 3: Phân Tích Đa Biến & Mối Tương Quan](#section-3)
4. [Section 4: Làm Sạch & Tiền Xử Lý Dữ Liệu](#section-4)

---

> **Lưu ý:** Vì đây là dataset NLP (dữ liệu văn bản), quy trình EDA được điều chỉnh phù hợp:  
> - Trích xuất các đặc trưng số (Feature Engineering) từ text: độ dài email, số từ, số ký tự đặc biệt...  
> - Phân tích tần suất từ, Word Cloud, và phân phối độ dài  
> - Áp dụng NLP preprocessing: lowercasing, loại bỏ stopwords, tokenization

<a id='section-1'></a>
# SECTION 1: Khởi tạo & Thống kê Mô tả Tổng quan
## (Overview & Basic Inspection)

### 1.1 Import các thư viện cần thiết

In [ ]:
# ============================================================
# 1.1 — Import đầy đủ các thư viện cần thiết
# ============================================================

# --- Thư viện xử lý dữ liệu ---
import pandas as pd                    # Xử lý dữ liệu dạng bảng (DataFrame)
import numpy as np                     # Tính toán số học, mảng

# --- Thư viện trực quan hóa ---
import matplotlib.pyplot as plt        # Vẽ biểu đồ cơ bản
import matplotlib.ticker as ticker     # Định dạng trục biểu đồ
import seaborn as sns                  # Vẽ biểu đồ thống kê nâng cao

# --- Thư viện xử lý văn bản (NLP) ---
import re                              # Biểu thức chính quy (Regular Expression)
import string                          # Các hằng số chuỗi ký tự
from collections import Counter        # Đếm tần suất phần tử

# --- Thư viện Machine Learning ---
from sklearn.model_selection import train_test_split   # Chia tập Train/Test
from sklearn.preprocessing import LabelEncoder         # Mã hóa nhãn
from sklearn.preprocessing import StandardScaler       # Chuẩn hóa dữ liệu

# --- Cấu hình hiển thị ---
import warnings
warnings.filterwarnings('ignore')      # Tắt các cảnh báo không cần thiết

# Cấu hình pandas hiển thị đầy đủ
pd.set_option('display.max_columns', None)       # Hiện tất cả các cột
pd.set_option('display.max_colwidth', 200)        # Hiện tối đa 200 ký tự mỗi cột
pd.set_option('display.float_format', '{:.4f}'.format)  # Định dạng số thập phân 4 chữ số

# Cấu hình matplotlib hiển thị inline trong notebook
%matplotlib inline

# Thiết lập style cho seaborn — giao diện hiện đại, sạch sẽ
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)          # Kích thước mặc định biểu đồ
plt.rcParams['figure.dpi'] = 100                   # Độ phân giải biểu đồ
plt.rcParams['axes.titlesize'] = 14                # Kích thước tiêu đề
plt.rcParams['axes.labelsize'] = 12                # Kích thước nhãn trục

print("✅ Import thư viện thành công!")
print(f"   📦 pandas   : {pd.__version__}")
print(f"   📦 numpy    : {np.__version__}")
print(f"   📦 seaborn  : {sns.__version__}")

### 1.2 Đọc dữ liệu & Kiểm tra nhanh

In [ ]:
# ============================================================
# 1.2 — Đọc file CSV và kiểm tra nhanh dữ liệu
# ============================================================

# Đường dẫn tới file CSV gốc
DATA_PATH = '../data/raw/phishing-email-dataset.csv'

# Đọc file CSV vào DataFrame
# - index_col=0: sử dụng cột đầu tiên ('Unnamed: 0') làm index thay vì để nó thành cột riêng
df_raw = pd.read_csv(DATA_PATH, index_col=0)

# Tạo bản sao để thao tác, giữ nguyên bản gốc cho đối chiếu
df = df_raw.copy()

print(f"📂 Đã tải dataset thành công từ: {DATA_PATH}")
print(f"📊 Kích thước dataset: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"💾 Bộ nhớ sử dụng: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

In [ ]:
# --- Hiển thị 5 dòng đầu tiên ---
print("="*80)
print("📋 5 DÒNG ĐẦU TIÊN (head)")
print("="*80)
display(df.head())

In [ ]:
# --- Hiển thị 5 dòng cuối cùng ---
print("="*80)
print("📋 5 DÒNG CUỐI CÙNG (tail)")
print("="*80)
display(df.tail())

### 1.3 Kiểm tra kiểu dữ liệu & Cấu trúc

In [ ]:
# ============================================================
# 1.3 — Kiểm tra chi tiết kiểu dữ liệu & cấu trúc
# ============================================================

print("="*80)
print("🔍 THÔNG TIN CẤU TRÚC DATASET (info)")
print("="*80)
df.info()

print("\n" + "="*80)
print("📊 KIỂU DỮ LIỆU CỦA TỪNG CỘT")
print("="*80)

# Tạo bảng tóm tắt kiểu dữ liệu
dtype_summary = pd.DataFrame({
    'Tên cột': df.columns,
    'Kiểu dữ liệu': df.dtypes.values,
    'Số giá trị duy nhất': df.nunique().values,
    'Giá trị mẫu': [df[col].iloc[0] if len(df) > 0 else 'N/A' for col in df.columns]
})
display(dtype_summary)

### 1.4 Phân tích dữ liệu khuyết thiếu (Missing Values)

In [ ]:
# ============================================================
# 1.4 — Thống kê chi tiết dữ liệu khuyết thiếu
# ============================================================

def analyze_missing_values(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Hàm phân tích và tạo bảng thống kê dữ liệu khuyết thiếu.
    
    Parameters:
        dataframe: DataFrame cần phân tích
    Returns:
        DataFrame chứa thông tin missing values, sắp xếp giảm dần theo tỷ lệ
    """
    # Đếm số giá trị null trong mỗi cột
    missing_count = dataframe.isnull().sum()
    
    # Tính tỷ lệ phần trăm khuyết thiếu
    missing_pct = (missing_count / len(dataframe)) * 100
    
    # Tạo bảng tổng hợp
    missing_table = pd.DataFrame({
        'Tên cột': missing_count.index,
        'Số lượng khuyết': missing_count.values,
        'Tỷ lệ (%)': missing_pct.values,
        'Kiểu dữ liệu': dataframe.dtypes.values
    })
    
    # Sắp xếp theo tỷ lệ khuyết giảm dần
    missing_table = missing_table.sort_values('Tỷ lệ (%)', ascending=False)
    missing_table = missing_table.reset_index(drop=True)
    
    return missing_table

# Gọi hàm và hiển thị kết quả
missing_df = analyze_missing_values(df)

print("="*80)
print("❓ BẢNG THỐNG KÊ DỮ LIỆU KHUYẾT THIẾU")
print("="*80)
display(missing_df)

# Tổng kết nhanh
total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]
print(f"\n📊 Tổng số ô dữ liệu     : {total_cells:,}")
print(f"❌ Tổng số ô bị khuyết    : {total_missing:,}")
print(f"📉 Tỷ lệ khuyết tổng thể : {(total_missing/total_cells)*100:.4f}%")

In [ ]:
# --- Trực quan hóa dữ liệu khuyết thiếu bằng biểu đồ cột ---

# Chỉ vẽ nếu có cột bị khuyết
cols_with_missing = missing_df[missing_df['Số lượng khuyết'] > 0]

if len(cols_with_missing) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    bars = ax.bar(
        cols_with_missing['Tên cột'],
        cols_with_missing['Tỷ lệ (%)'],
        color=sns.color_palette('Reds_r', len(cols_with_missing)),
        edgecolor='black',
        linewidth=0.5
    )
    
    # Thêm nhãn giá trị trên mỗi cột
    for bar, count in zip(bars, cols_with_missing['Số lượng khuyết']):
        ax.text(
            bar.get_x() + bar.get_width() / 2.,
            bar.get_height() + 0.3,
            f'{count:,}\n({bar.get_height():.2f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold'
        )
    
    ax.set_title('📊 Tỷ Lệ Dữ Liệu Khuyết Thiếu Theo Cột', fontsize=14, fontweight='bold')
    ax.set_ylabel('Tỷ lệ khuyết (%)')
    ax.set_xlabel('Tên cột')
    plt.tight_layout()
    plt.show()
else:
    print("✅ Không có cột nào bị khuyết dữ liệu!")

### 1.5 Kiểm tra dữ liệu trùng lặp (Duplicates)

In [ ]:
# ============================================================
# 1.5 — Kiểm tra và thống kê dữ liệu trùng lặp
# ============================================================

# Đếm số dòng trùng lặp hoàn toàn (tất cả cột giống nhau)
num_duplicates = df.duplicated().sum()
pct_duplicates = (num_duplicates / len(df)) * 100

print("="*80)
print("🔄 KIỂM TRA DỮ LIỆU TRÙNG LẶP")
print("="*80)
print(f"   Tổng số dòng            : {len(df):,}")
print(f"   Số dòng trùng lặp       : {num_duplicates:,}")
print(f"   Tỷ lệ trùng lặp        : {pct_duplicates:.2f}%")

# Hiển thị một số dòng trùng lặp mẫu (nếu có)
if num_duplicates > 0:
    print(f"\n📋 Mẫu 5 dòng bị trùng lặp:")
    display(df[df.duplicated(keep='first')].head())

### 1.6 Thống kê mô tả (Descriptive Statistics)

In [ ]:
# ============================================================
# 1.6 — Thống kê mô tả cho biến phân loại (Categorical)
# ============================================================

print("="*80)
print("📊 THỐNG KÊ MÔ TẢ — BIẾN PHÂN LOẠI (Categorical)")
print("="*80)

# Sử dụng include='O' (Object) để chỉ thống kê biến dạng chuỗi
display(df.describe(include='O'))

### 1.7 Feature Engineering — Trích xuất đặc trưng số từ văn bản

> Vì dataset chủ yếu là dữ liệu text, ta cần trích xuất các đặc trưng số (numerical features) từ nội dung email để phục vụ phân tích thống kê và trực quan hóa.

In [ ]:
# ============================================================
# 1.7 — Trích xuất đặc trưng số từ cột "Email Text"
# ============================================================

def extract_text_features(dataframe: pd.DataFrame, text_col: str = 'Email Text') -> pd.DataFrame:
    """
    Hàm trích xuất các đặc trưng thống kê từ cột văn bản.
    
    Parameters:
        dataframe: DataFrame gốc
        text_col: Tên cột chứa nội dung văn bản
    Returns:
        DataFrame đã được bổ sung các cột đặc trưng mới
    """
    df_feat = dataframe.copy()
    
    # Điền chuỗi rỗng cho các giá trị NaN để tránh lỗi khi xử lý chuỗi
    text_series = df_feat[text_col].fillna('')
    
    # 1. Độ dài email (số ký tự)
    df_feat['char_count'] = text_series.str.len()
    
    # 2. Số từ trong email
    df_feat['word_count'] = text_series.str.split().str.len().fillna(0).astype(int)
    
    # 3. Độ dài trung bình mỗi từ
    df_feat['avg_word_length'] = df_feat.apply(
        lambda row: np.mean([len(w) for w in str(row[text_col]).split()]) 
        if row['word_count'] > 0 else 0,
        axis=1
    )
    
    # 4. Số câu trong email (ước lượng bằng dấu chấm câu)
    df_feat['sentence_count'] = text_series.str.count(r'[.!?]+').clip(lower=1)
    
    # 5. Số ký tự viết HOA
    df_feat['uppercase_count'] = text_series.str.count(r'[A-Z]')
    
    # 6. Tỷ lệ ký tự viết HOA / tổng ký tự
    df_feat['uppercase_ratio'] = (
        df_feat['uppercase_count'] / df_feat['char_count'].replace(0, 1)
    )
    
    # 7. Số ký tự đặc biệt (không phải chữ cái hoặc số)
    df_feat['special_char_count'] = text_series.str.count(r'[^a-zA-Z0-9\s]')
    
    # 8. Số lượng URL trong email
    df_feat['url_count'] = text_series.str.count(
        r'https?://[^\s]+|www\.[^\s]+'
    )
    
    # 9. Số lượng địa chỉ email trong nội dung
    df_feat['email_addr_count'] = text_series.str.count(
        r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    )
    
    # 10. Số lượng chữ số (digits)
    df_feat['digit_count'] = text_series.str.count(r'\d')
    
    return df_feat

# Áp dụng hàm trích xuất đặc trưng
df = extract_text_features(df)

print("✅ Đã trích xuất 10 đặc trưng số từ văn bản email!")
print(f"📊 Kích thước dataset mới: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print("\n📋 Danh sách các cột hiện tại:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")

In [ ]:
# --- Thống kê mô tả cho các đặc trưng số vừa trích xuất ---

print("="*80)
print("📊 THỐNG KÊ MÔ TẢ — BIẾN SỐ (Numerical Features)")
print("="*80)

# Lấy danh sách các cột số (numeric)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"   Số lượng biến số: {len(numeric_cols)}")
print(f"   Danh sách: {numeric_cols}\n")

# Hiển thị thống kê describe() với transpose cho dễ đọc
display(df[numeric_cols].describe().T)

---

**Nhận xét của Data Scientist:**

> _[Ghi chú phân tích sau khi chạy Section 1 — Ví dụ: Kích thước dataset, tỷ lệ missing, kiểu dữ liệu có hợp lý không, có cần chuyển đổi kiểu dữ liệu nào không, nhận xét sơ bộ về phân phối các đặc trưng...]_

---

<a id='section-2'></a>
# SECTION 2: Phân Tích Đơn Biến (Univariate Analysis)

### 2.1 Phân tích biến mục tiêu — `Email Type` (Target Variable)

> Kiểm tra tỷ lệ phân phối giữa 2 lớp: **Safe Email** vs **Phishing Email**  
> → Đánh giá mức độ **mất cân bằng dữ liệu (Class Imbalance)**

In [ ]:
# ============================================================
# 2.1 — Phân tích biến mục tiêu: Email Type
# ============================================================

TARGET_COL = 'Email Type'  # Tên cột mục tiêu

# Đếm tần suất mỗi lớp
target_counts = df[TARGET_COL].value_counts()
target_pct = df[TARGET_COL].value_counts(normalize=True) * 100

print("="*80)
print("🎯 PHÂN PHỐI BIẾN MỤC TIÊU — Email Type")
print("="*80)

# Tạo bảng thống kê
target_summary = pd.DataFrame({
    'Lớp (Class)': target_counts.index,
    'Số lượng': target_counts.values,
    'Tỷ lệ (%)': target_pct.values
})
display(target_summary)

# Tính tỷ lệ mất cân bằng
imbalance_ratio = target_counts.max() / target_counts.min()
print(f"\n⚖️  Tỷ lệ mất cân bằng (Imbalance Ratio): {imbalance_ratio:.2f}:1")

if imbalance_ratio > 3:
    print("⚠️  CẢNH BÁO: Dataset bị mất cân bằng NGHIÊM TRỌNG! Cần xem xét kỹ thuật xử lý (SMOTE, class_weight, ...)")
elif imbalance_ratio > 1.5:
    print("🔶 Dataset có mất cân bằng nhẹ. Cần theo dõi hiệu năng trên lớp thiểu số.")
else:
    print("✅ Dataset tương đối cân bằng.")

In [ ]:
# --- Trực quan hóa phân phối Target ---

# Bảng màu riêng cho từng lớp
TARGET_COLORS = {'Safe Email': '#2ecc71', 'Phishing Email': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Biểu đồ 1: Countplot (biểu đồ cột) ---
ax1 = axes[0]
bars = ax1.bar(
    target_counts.index,
    target_counts.values,
    color=[TARGET_COLORS[label] for label in target_counts.index],
    edgecolor='black',
    linewidth=0.8,
    width=0.5
)

# Thêm nhãn giá trị
for bar, count, pct in zip(bars, target_counts.values, target_pct.values):
    ax1.text(
        bar.get_x() + bar.get_width() / 2.,
        bar.get_height() + len(df) * 0.01,
        f'{count:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax1.set_title('📊 Phân Phối Biến Mục Tiêu (Countplot)', fontweight='bold')
ax1.set_ylabel('Số lượng email')
ax1.set_xlabel('Loại Email')

# --- Biểu đồ 2: Pie Chart ---
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    target_counts.values,
    labels=target_counts.index,
    autopct='%1.1f%%',
    colors=[TARGET_COLORS[label] for label in target_counts.index],
    startangle=90,
    explode=[0.03] * len(target_counts),
    shadow=True,
    textprops={'fontsize': 12}
)
# Tô đậm phần trăm
for autotext in autotexts:
    autotext.set_fontweight('bold')

ax2.set_title('🥧 Tỷ Lệ Các Lớp (Pie Chart)', fontweight='bold')

plt.suptitle('PHÂN TÍCH BIẾN MỤC TIÊU — Email Type', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.2 Phân tích phân phối các biến số (Numerical Features)

> Vẽ tự động **Histogram + KDE** và **Boxplot** cho tất cả biến số để phát hiện:  
> - Phân phối lệch (Skewness)  
> - Giá trị ngoại lai (Outliers)

In [ ]:
# ============================================================
# 2.2 — Hàm vẽ Histogram + KDE + Boxplot cho biến số
# ============================================================

def plot_numeric_distributions(dataframe: pd.DataFrame, 
                                numeric_columns: list,
                                cols_per_row: int = 2) -> None:
    """
    Hàm tự động vẽ cặp biểu đồ (Histogram+KDE, Boxplot) cho mỗi biến số.
    
    Parameters:
        dataframe: DataFrame chứa dữ liệu
        numeric_columns: Danh sách tên các cột số cần vẽ
        cols_per_row: Số cặp biểu đồ trên mỗi hàng
    """
    n_features = len(numeric_columns)
    
    # Mỗi feature chiếm 1 hàng với 2 cột (Histogram | Boxplot)
    fig, axes = plt.subplots(
        nrows=n_features, ncols=2,
        figsize=(16, 4 * n_features)
    )
    
    # Đảm bảo axes luôn là 2D array
    if n_features == 1:
        axes = axes.reshape(1, -1)
    
    # Bảng màu đa dạng
    colors = sns.color_palette('husl', n_features)
    
    for i, col in enumerate(numeric_columns):
        # --- Histogram + KDE (cột trái) ---
        ax_hist = axes[i, 0]
        sns.histplot(
            data=dataframe, x=col, kde=True,
            color=colors[i], ax=ax_hist,
            bins=50, edgecolor='white', linewidth=0.5
        )
        
        # Thêm thông tin skewness & kurtosis
        skew_val = dataframe[col].skew()
        kurt_val = dataframe[col].kurtosis()
        ax_hist.axvline(dataframe[col].median(), color='red', linestyle='--', 
                        linewidth=1.5, label=f'Median: {dataframe[col].median():.2f}')
        ax_hist.axvline(dataframe[col].mean(), color='blue', linestyle='-.',
                        linewidth=1.5, label=f'Mean: {dataframe[col].mean():.2f}')
        ax_hist.set_title(f'📊 Histogram + KDE: {col}\n(Skew={skew_val:.2f}, Kurt={kurt_val:.2f})',
                         fontsize=11, fontweight='bold')
        ax_hist.legend(fontsize=9)
        
        # --- Boxplot (cột phải) ---
        ax_box = axes[i, 1]
        sns.boxplot(
            data=dataframe, x=col,
            color=colors[i], ax=ax_box,
            flierprops=dict(marker='o', markersize=3, alpha=0.5)
        )
        
        # Đếm số outliers bằng IQR
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        n_outliers = ((dataframe[col] < Q1 - 1.5 * IQR) | (dataframe[col] > Q3 + 1.5 * IQR)).sum()
        pct_outliers = (n_outliers / len(dataframe)) * 100
        
        ax_box.set_title(
            f'📦 Boxplot: {col}\n(Outliers: {n_outliers:,} — {pct_outliers:.2f}%)',
            fontsize=11, fontweight='bold'
        )
    
    plt.suptitle(
        'PHÂN PHỐI CÁC BIẾN SỐ (Numerical Features)',
        fontsize=16, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()

# Gọi hàm vẽ cho tất cả biến số
plot_numeric_distributions(df, numeric_cols)

In [ ]:
# --- Bảng tóm tắt Skewness và Outliers cho tất cả biến số ---

def summarize_numeric_features(dataframe: pd.DataFrame, 
                                numeric_columns: list) -> pd.DataFrame:
    """
    Hàm tạo bảng tóm tắt skewness, kurtosis, và số lượng outliers (IQR).
    """
    summary_rows = []
    
    for col in numeric_columns:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        n_outliers = ((dataframe[col] < lower_bound) | (dataframe[col] > upper_bound)).sum()
        
        summary_rows.append({
            'Feature': col,
            'Mean': dataframe[col].mean(),
            'Median': dataframe[col].median(),
            'Std': dataframe[col].std(),
            'Skewness': dataframe[col].skew(),
            'Kurtosis': dataframe[col].kurtosis(),
            'IQR': IQR,
            'Outliers': n_outliers,
            'Outlier %': (n_outliers / len(dataframe)) * 100
        })
    
    return pd.DataFrame(summary_rows)

outlier_summary = summarize_numeric_features(df, numeric_cols)

print("="*80)
print("📊 BẢNG TÓM TẮT: SKEWNESS & OUTLIERS (IQR Method)")
print("="*80)
display(outlier_summary)

### 2.3 Phân tích phân phối theo Target — So sánh Safe vs Phishing

> So sánh phân phối các đặc trưng giữa 2 lớp email để tìm đặc trưng phân biệt mạnh.

In [ ]:
# ============================================================
# 2.3 — Histogram phân tách theo Target cho mỗi biến số
# ============================================================

def plot_distributions_by_target(dataframe: pd.DataFrame,
                                  numeric_columns: list,
                                  target_col: str = 'Email Type',
                                  palette: dict = None) -> None:
    """
    Hàm vẽ KDE plot so sánh phân phối giữa các lớp của biến mục tiêu.
    """
    if palette is None:
        palette = {'Safe Email': '#2ecc71', 'Phishing Email': '#e74c3c'}
    
    n_features = len(numeric_columns)
    n_cols = 3  # 3 biểu đồ mỗi hàng
    n_rows = (n_features + n_cols - 1) // n_cols  # Làm tròn lên
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    axes = axes.flatten()  # Flatten để dễ duyệt
    
    for i, col in enumerate(numeric_columns):
        ax = axes[i]
        
        # Vẽ KDE cho mỗi lớp
        for label, color in palette.items():
            subset = dataframe[dataframe[target_col] == label][col].dropna()
            if len(subset) > 0:
                sns.kdeplot(
                    data=subset, ax=ax,
                    color=color, fill=True, alpha=0.3,
                    label=label, linewidth=2
                )
        
        ax.set_title(f'{col}', fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
    
    # Ẩn các axes thừa
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle(
        '📊 PHÂN PHỐI CÁC ĐẶC TRƯNG THEO LOẠI EMAIL (Safe vs Phishing)',
        fontsize=16, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()

# Gọi hàm vẽ
plot_distributions_by_target(df, numeric_cols, TARGET_COL, TARGET_COLORS)

### 2.4 Phân tích độ dài email — Boxplot theo Target

In [ ]:
# ============================================================
# 2.4 — Boxplot so sánh các đặc trưng chính theo Target
# ============================================================

# Chọn 4 đặc trưng quan trọng nhất để so sánh chi tiết
key_features = ['char_count', 'word_count', 'uppercase_ratio', 'url_count']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(key_features):
    ax = axes[i]
    sns.boxplot(
        data=df, x=TARGET_COL, y=col,
        palette=TARGET_COLORS, ax=ax,
        flierprops=dict(marker='o', markersize=2, alpha=0.3)
    )
    ax.set_title(f'📦 {col} theo Email Type', fontweight='bold')
    ax.set_xlabel('')

plt.suptitle(
    'SO SÁNH CÁC ĐẶC TRƯNG CHÍNH GIỮA SAFE vs PHISHING',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

### 2.5 Phân tích Top từ xuất hiện nhiều nhất (Word Frequency Analysis)

In [ ]:
# ============================================================
# 2.5 — Phân tích tần suất từ cho mỗi loại email
# ============================================================

# Danh sách stopwords cơ bản (tiếng Anh)
STOP_WORDS = set([
    'the', 'a', 'an', 'is', 'it', 'to', 'in', 'of', 'and', 'for',
    'on', 'that', 'this', 'with', 'you', 'are', 'was', 'be', 'as',
    'at', 'by', 'we', 'or', 'not', 'from', 'but', 'have', 'has',
    'had', 'do', 'does', 'did', 'will', 'would', 'can', 'could',
    'should', 'may', 'might', 'shall', 'i', 'he', 'she', 'they',
    'me', 'him', 'her', 'us', 'them', 'my', 'your', 'his', 'its',
    'our', 'their', 'what', 'which', 'who', 'whom', 'if', 'then',
    'than', 'so', 'no', 'all', 'each', 'any', 'both', 'more',
    'most', 'other', 'some', 'such', 'only', 'own', 'same', 'too',
    'very', 'just', 'because', 'how', 'when', 'where', 'there',
    'here', 'about', 'up', 'out', 'been', 'being', 'these', 'those',
    'also', 'after', 'before', 'between', 'through', 'during',
    'above', 'below', 'into', 'over', 'under', 'again', 'further',
    'once', 'am', 'were', 'don', 't', 's', 're', 've', 'll', 'd', 'm'
])


def get_top_words(texts: pd.Series, n_top: int = 20, 
                  stop_words: set = STOP_WORDS) -> list:
    """
    Hàm trích xuất top N từ xuất hiện nhiều nhất (sau khi loại bỏ stopwords).
    
    Parameters:
        texts: Series chứa nội dung văn bản
        n_top: Số lượng từ top cần trả về
        stop_words: Tập hợp các stopwords cần loại bỏ
    Returns:
        Danh sách tuple (từ, tần suất)
    """
    # Gộp tất cả text, chuyển thành chữ thường
    all_text = ' '.join(texts.dropna().astype(str)).lower()
    
    # Chỉ giữ lại chữ cái và khoảng trắng
    all_text = re.sub(r'[^a-z\s]', '', all_text)
    
    # Tách từ và loại bỏ stopwords + từ quá ngắn
    words = [w for w in all_text.split() if w not in stop_words and len(w) > 2]
    
    # Đếm tần suất và trả về top N
    return Counter(words).most_common(n_top)


# Lấy top words cho mỗi loại email
safe_texts = df[df[TARGET_COL] == 'Safe Email']['Email Text']
phishing_texts = df[df[TARGET_COL] == 'Phishing Email']['Email Text']

top_safe = get_top_words(safe_texts, n_top=20)
top_phishing = get_top_words(phishing_texts, n_top=20)

# --- Vẽ biểu đồ tần suất từ ---
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Safe Email
words_safe, counts_safe = zip(*top_safe)
axes[0].barh(words_safe[::-1], counts_safe[::-1], color='#2ecc71', edgecolor='black', linewidth=0.5)
axes[0].set_title('✅ Top 20 Từ — Safe Email', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tần suất')

# Phishing Email
words_phish, counts_phish = zip(*top_phishing)
axes[1].barh(words_phish[::-1], counts_phish[::-1], color='#e74c3c', edgecolor='black', linewidth=0.5)
axes[1].set_title('🚨 Top 20 Từ — Phishing Email', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Tần suất')

plt.suptitle('PHÂN TÍCH TẦN SUẤT TỪ THEO LOẠI EMAIL', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

**Nhận xét của Data Scientist:**

> _[Ghi chú phân tích sau khi chạy Section 2 — Ví dụ: Tỷ lệ cân bằng giữa 2 lớp, phân phối nào bị lệch mạnh, biến nào có nhiều outliers, có những từ khóa phân biệt nào giữa Safe và Phishing email, ...]_

---

<a id='section-3'></a>
# SECTION 3: Phân Tích Đa Biến & Mối Tương Quan
## (Multivariate & Correlation Analysis)

### 3.1 Ma trận tương quan (Correlation Matrix Heatmap)

> Phát hiện hiện tượng **đa cộng tuyến (Multicollinearity)** giữa các đặc trưng số.

In [ ]:
# ============================================================
# 3.1 — Ma trận tương quan giữa các biến số
# ============================================================

# Tính ma trận tương quan Pearson
corr_matrix = df[numeric_cols].corr()

# Tạo mask cho nửa trên ma trận (loại bỏ trùng lặp)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 10))

sns.heatmap(
    corr_matrix,
    mask=mask,                    # Chỉ hiển thị nửa dưới tam giác
    annot=True,                   # Hiển thị giá trị tương quan
    fmt='.2f',                    # Định dạng 2 chữ số thập phân
    cmap='RdBu_r',               # Bảng màu: đỏ (âm) → trắng (0) → xanh (dương)
    vmin=-1, vmax=1,             # Giới hạn thang màu [-1, 1]
    center=0,                     # Trung tâm màu tại 0
    square=True,                  # Ô vuông cho đẹp
    linewidths=0.5,              # Đường viền giữa các ô
    cbar_kws={'label': 'Hệ số tương quan Pearson'},
    ax=ax
)

ax.set_title(
    '🔥 MA TRẬN TƯƠNG QUAN (Correlation Matrix Heatmap)\nPhát hiện đa cộng tuyến giữa các đặc trưng',
    fontsize=14, fontweight='bold', pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
# --- Liệt kê các cặp đặc trưng có tương quan CAO (> 0.7 hoặc < -0.7) ---

def find_high_correlations(corr_matrix: pd.DataFrame, threshold: float = 0.7) -> pd.DataFrame:
    """
    Hàm tìm các cặp đặc trưng có tương quan cao (vượt ngưỡng threshold).
    
    Parameters:
        corr_matrix: Ma trận tương quan
        threshold: Ngưỡng tương quan (mặc định 0.7)
    Returns:
        DataFrame chứa các cặp có tương quan cao
    """
    pairs = []
    
    # Duyệt qua nửa dưới ma trận tam giác (tránh trùng lặp)
    for i in range(len(corr_matrix.columns)):
        for j in range(i + 1, len(corr_matrix.columns)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) >= threshold:
                pairs.append({
                    'Feature 1': corr_matrix.columns[i],
                    'Feature 2': corr_matrix.columns[j],
                    'Correlation': corr_val,
                    'Abs Correlation': abs(corr_val)
                })
    
    result = pd.DataFrame(pairs).sort_values('Abs Correlation', ascending=False)
    return result.reset_index(drop=True)

high_corr = find_high_correlations(corr_matrix, threshold=0.7)

print("="*80)
print(f"⚠️  CÁC CẶP ĐẶC TRƯNG CÓ TƯƠNG QUAN CAO (|r| ≥ 0.7)")
print("="*80)

if len(high_corr) > 0:
    display(high_corr)
    print("\n💡 Gợi ý: Xem xét loại bỏ 1 trong 2 đặc trưng ở mỗi cặp để tránh đa cộng tuyến.")
else:
    print("✅ Không có cặp đặc trưng nào có tương quan cao vượt ngưỡng 0.7.")

### 3.2 Tương quan giữa các đặc trưng và biến mục tiêu (Target Correlation)

In [ ]:
# ============================================================
# 3.2 — Tương quan giữa các đặc trưng và biến mục tiêu
# ============================================================

# Mã hóa biến mục tiêu thành số để tính tương quan
# Safe Email = 0, Phishing Email = 1
target_encoded = (df[TARGET_COL] == 'Phishing Email').astype(int)

# Tính hệ số tương quan Point-Biserial giữa từng đặc trưng số và Target
target_corr = df[numeric_cols].corrwith(target_encoded).sort_values(ascending=True)

# Vẽ biểu đồ cột ngang
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in target_corr.values]

bars = ax.barh(
    target_corr.index,
    target_corr.values,
    color=colors,
    edgecolor='black',
    linewidth=0.5
)

# Thêm nhãn giá trị
for bar, val in zip(bars, target_corr.values):
    ax.text(
        val + (0.005 if val > 0 else -0.005),
        bar.get_y() + bar.get_height() / 2.,
        f'{val:.3f}',
        ha='left' if val > 0 else 'right',
        va='center', fontsize=10, fontweight='bold'
    )

# Đường tham chiếu tại 0
ax.axvline(x=0, color='black', linewidth=0.8)

ax.set_title(
    '🎯 TƯƠNG QUAN GIỮA CÁC ĐẶC TRƯNG VÀ BIẾN MỤC TIÊU\n'
    '(Đỏ = Tương quan dương với Phishing | Xanh = Tương quan âm)',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Hệ số tương quan (Point-Biserial)')

plt.tight_layout()
plt.show()

### 3.3 Pairplot — Các đặc trưng quan trọng nhất

In [ ]:
# ============================================================
# 3.3 — Pairplot cho top 4 đặc trưng có tương quan mạnh nhất với Target
# ============================================================

# Chọn top 4 đặc trưng có tương quan tuyệt đối lớn nhất với Target
top_features = target_corr.abs().nlargest(4).index.tolist()

print(f"📊 Top 4 đặc trưng tương quan mạnh nhất với Target: {top_features}")

# Lấy mẫu con để Pairplot (dataset lớn thì vẽ cả sẽ rất chậm)
SAMPLE_SIZE = 5000  # Lấy 5000 mẫu ngẫu nhiên
df_sample = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42)

# Vẽ Pairplot
pair_plot = sns.pairplot(
    df_sample,
    vars=top_features,
    hue=TARGET_COL,
    palette=TARGET_COLORS,
    diag_kind='kde',        # Đường chéo dùng KDE thay vì histogram
    plot_kws={'alpha': 0.4, 's': 15},  # Điểm mờ, nhỏ
    diag_kws={'fill': True, 'alpha': 0.5}
)

pair_plot.fig.suptitle(
    'PAIRPLOT — Top Đặc Trưng Tương Quan Mạnh Nhất Với Target',
    fontsize=14, fontweight='bold', y=1.02
)
plt.show()

### 3.4 Thống kê trung bình các đặc trưng theo Target

In [ ]:
# ============================================================
# 3.4 — So sánh giá trị trung bình các đặc trưng giữa 2 lớp
# ============================================================

# Tính mean, median theo nhóm Target
group_stats = df.groupby(TARGET_COL)[numeric_cols].agg(['mean', 'median'])

print("="*80)
print("📊 SO SÁNH TRUNG BÌNH & TRUNG VỊ CÁC ĐẶC TRƯNG THEO LOẠI EMAIL")
print("="*80)
display(group_stats)

---

**Nhận xét của Data Scientist:**

> _[Ghi chú phân tích sau khi chạy Section 3 — Ví dụ: Cặp đặc trưng nào có đa cộng tuyến cao, đặc trưng nào phân biệt mạnh nhất giữa Safe và Phishing, có nên loại bỏ đặc trưng nào không, các pattern thú vị phát hiện được từ pairplot, ...]_

---

<a id='section-4'></a>
# SECTION 4: Làm Sạch & Tiền Xử Lý Dữ Liệu
## (Data Cleaning & Preprocessing)

### 4.1 Xử lý giá trị khuyết thiếu (Missing Values Treatment)

In [ ]:
# ============================================================
# 4.1 — Xử lý giá trị khuyết thiếu
# ============================================================

# Tạo bản sao để xử lý, giữ nguyên df cho đối chiếu
df_clean = df.copy()

print("="*80)
print("🧹 XỬ LÝ GIÁ TRỊ KHUYẾT THIẾU")
print("="*80)

# --- Kiểm tra missing values trước khi xử lý ---
print("\n📋 TRƯỚC KHI XỬ LÝ:")
print(df_clean.isnull().sum())

# --- Chiến lược xử lý ---
# Cột 'Email Text': Điền chuỗi rỗng nếu thiếu (vì đây là cột text chính)
# Các cột số: Điền median (do phân phối lệch, median ổn định hơn mean)

# Xử lý cột text: điền chuỗi rỗng
if df_clean['Email Text'].isnull().sum() > 0:
    n_missing_text = df_clean['Email Text'].isnull().sum()
    df_clean['Email Text'] = df_clean['Email Text'].fillna('')
    print(f"\n   ✅ 'Email Text': Đã điền {n_missing_text} giá trị NaN bằng chuỗi rỗng.")

# Xử lý cột target: drop dòng nếu thiếu (không thể suy luận nhãn)
if df_clean[TARGET_COL].isnull().sum() > 0:
    n_before = len(df_clean)
    df_clean = df_clean.dropna(subset=[TARGET_COL])
    n_dropped = n_before - len(df_clean)
    print(f"   ❌ '{TARGET_COL}': Đã loại bỏ {n_dropped} dòng thiếu nhãn mục tiêu.")

# Xử lý các cột số: điền median
numeric_cols_current = df_clean.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols_current:
    n_missing = df_clean[col].isnull().sum()
    if n_missing > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"   ✅ '{col}': Đã điền {n_missing} giá trị NaN bằng median ({median_val:.2f}).")

# --- Kiểm tra sau khi xử lý ---
print("\n📋 SAU KHI XỬ LÝ:")
print(df_clean.isnull().sum())
print(f"\n📊 Kích thước dataset sau xử lý: {df_clean.shape[0]:,} dòng × {df_clean.shape[1]} cột")

### 4.2 Xử lý dữ liệu trùng lặp (Duplicate Removal)

In [ ]:
# ============================================================
# 4.2 — Xử lý dữ liệu trùng lặp
# ============================================================

print("="*80)
print("🔄 XỬ LÝ DỮ LIỆU TRÙNG LẶP")
print("="*80)

n_before = len(df_clean)
n_dups = df_clean.duplicated().sum()

print(f"   Số dòng trước khi xử lý   : {n_before:,}")
print(f"   Số dòng trùng lặp phát hiện: {n_dups:,}")

if n_dups > 0:
    # Loại bỏ dòng trùng lặp, giữ lại dòng đầu tiên
    df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)
    print(f"   ✅ Đã loại bỏ {n_dups:,} dòng trùng lặp.")
    print(f"   Số dòng còn lại           : {len(df_clean):,}")
else:
    print("   ✅ Không có dòng trùng lặp!")

### 4.3 Phát hiện và xử lý Outliers (Giá trị ngoại lai)

In [ ]:
# ============================================================
# 4.3 — Phát hiện và xử lý Outliers bằng phương pháp IQR
# ============================================================

def detect_and_cap_outliers_iqr(dataframe: pd.DataFrame, 
                                 columns: list,
                                 multiplier: float = 1.5,
                                 method: str = 'cap') -> pd.DataFrame:
    """
    Hàm phát hiện và xử lý outliers bằng phương pháp IQR.
    
    Parameters:
        dataframe: DataFrame cần xử lý
        columns: Danh sách cột số cần kiểm tra outliers
        multiplier: Hệ số nhân IQR (mặc định 1.5)
        method: 'cap' = cắt ngưỡng (winsorize), 'remove' = loại bỏ dòng
    Returns:
        DataFrame đã xử lý outliers
    """
    df_result = dataframe.copy()
    outlier_report = []
    
    for col in columns:
        Q1 = df_result[col].quantile(0.25)
        Q3 = df_result[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        
        # Đếm outliers
        n_lower = (df_result[col] < lower_bound).sum()
        n_upper = (df_result[col] > upper_bound).sum()
        n_total = n_lower + n_upper
        
        if n_total > 0:
            if method == 'cap':
                # Cắt ngưỡng (Winsorize): thay giá trị outlier bằng giới hạn IQR
                df_result[col] = df_result[col].clip(lower=lower_bound, upper=upper_bound)
            
            outlier_report.append({
                'Feature': col,
                'Lower Bound': lower_bound,
                'Upper Bound': upper_bound,
                'Outliers (dưới)': n_lower,
                'Outliers (trên)': n_upper,
                'Tổng Outliers': n_total,
                'Phương pháp': method.upper()
            })
    
    report_df = pd.DataFrame(outlier_report)
    return df_result, report_df


# Áp dụng: Cắt ngưỡng (cap) outliers cho các cột số
# Lưu ý: Chỉ xử lý outliers trên các cột đặc trưng, KHÔNG xử lý trên cột target
numeric_features = df_clean.select_dtypes(include=[np.number]).columns.tolist()

df_clean, outlier_report = detect_and_cap_outliers_iqr(
    df_clean, 
    columns=numeric_features,
    multiplier=1.5,
    method='cap'  # Sử dụng phương pháp cắt ngưỡng (Winsorize)
)

print("="*80)
print("📊 BÁO CÁO XỬ LÝ OUTLIERS (IQR × 1.5, Phương pháp: CAP/WINSORIZE)")
print("="*80)

if len(outlier_report) > 0:
    display(outlier_report)
    print("\n✅ Đã cắt ngưỡng (winsorize) tất cả outliers về giới hạn IQR.")
else:
    print("✅ Không phát hiện outliers đáng kể.")

### 4.4 Mã hóa biến phân loại (Categorical Encoding)

> Mã hóa biến mục tiêu `Email Type` thành giá trị nhị phân:  
> - `Safe Email` → **0**  
> - `Phishing Email` → **1**

In [ ]:
# ============================================================
# 4.4 — Mã hóa biến mục tiêu (Label Encoding)
# ============================================================

print("="*80)
print("🏷️  MÃ HÓA BIẾN MỤC TIÊU (Label Encoding)")
print("="*80)

# Sử dụng LabelEncoder của sklearn
le = LabelEncoder()

# Fit & Transform: Mã hóa Email Type
df_clean['label'] = le.fit_transform(df_clean[TARGET_COL])

# Hiển thị mapping
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"   📋 Mapping: {label_mapping}")

# Kiểm tra kết quả
print(f"\n   📊 Phân phối sau mã hóa:")
print(df_clean['label'].value_counts().to_string())

# Hiển thị mẫu
print("\n   📋 Mẫu sau mã hóa:")
display(df_clean[[TARGET_COL, 'label']].drop_duplicates())

### 4.5 Chuẩn bị đặc trưng & Chia tập Train/Test

In [ ]:
# ============================================================
# 4.5 — Chuẩn bị đặc trưng (Feature Matrix) và Chia tập Train/Test
# ============================================================

print("="*80)
print("✂️  CHIA TẬP TRAIN / TEST")
print("="*80)

# --- Xác định Feature Matrix (X) và Target Vector (y) ---

# Các cột đặc trưng số (loại bỏ cột text gốc, cột target gốc, và cột label)
feature_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in feature_cols if c != 'label']  # Loại bỏ cột nhãn

X = df_clean[feature_cols]  # Ma trận đặc trưng
y = df_clean['label']       # Vector nhãn mục tiêu

print(f"   📊 Feature Matrix (X) : {X.shape}")
print(f"   🎯 Target Vector  (y) : {y.shape}")
print(f"   📋 Các đặc trưng      : {feature_cols}")

# --- Chia Train/Test theo tỷ lệ 80/20 ---
# - stratify=y: Đảm bảo tỷ lệ lớp được giữ nguyên trong cả Train và Test
# - random_state=42: Đảm bảo tái lập được kết quả
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% cho Test
    random_state=42,      # Seed cố định để tái lập
    stratify=y            # Phân tầng theo nhãn
)

print(f"\n   📦 Tập Train : {X_train.shape[0]:,} mẫu ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   📦 Tập Test  : {X_test.shape[0]:,} mẫu ({X_test.shape[0]/len(X)*100:.1f}%)")

# Kiểm tra phân phối nhãn trong Train và Test
print("\n   ⚖️  Phân phối nhãn:")
train_dist = y_train.value_counts(normalize=True) * 100
test_dist = y_test.value_counts(normalize=True) * 100

dist_check = pd.DataFrame({
    'Train (%)': train_dist,
    'Test (%)': test_dist,
    'Gốc (%)': y.value_counts(normalize=True) * 100
})
display(dist_check)

### 4.6 Chuẩn hóa dữ liệu số (Feature Scaling)

> **QUAN TRỌNG**: Chỉ `fit` scaler trên tập **Train**, rồi `transform` trên cả Train và Test.  
> → Tránh **rò rỉ dữ liệu (Data Leakage)** từ tập Test vào quá trình tiền xử lý.

In [ ]:
# ============================================================
# 4.6 — Chuẩn hóa dữ liệu (Feature Scaling) — StandardScaler
# ============================================================

print("="*80)
print("📏 CHUẨN HÓA DỮ LIỆU (StandardScaler)")
print("="*80)

# Khởi tạo StandardScaler
# StandardScaler: z = (x - mean) / std → phân phối chuẩn (mean=0, std=1)
scaler = StandardScaler()

# ⚠️ QUAN TRỌNG: Chỉ FIT trên tập TRAIN để tránh Data Leakage
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),     # Fit + Transform trên Train
    columns=feature_cols,
    index=X_train.index
)

# Transform tập Test bằng scaler đã học từ Train
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),           # Chỉ Transform (KHÔNG fit) trên Test
    columns=feature_cols,
    index=X_test.index
)

print("   ✅ Đã chuẩn hóa StandardScaler thành công!")
print("   ⚠️  Scaler đã fit trên tập TRAIN → transform trên TEST (không rò rỉ dữ liệu).")

# --- Kiểm tra kết quả chuẩn hóa ---
print("\n📊 Thống kê tập TRAIN sau chuẩn hóa:")
display(X_train_scaled.describe().T[['mean', 'std', 'min', 'max']])

print("\n📊 Thống kê tập TEST sau chuẩn hóa:")
display(X_test_scaled.describe().T[['mean', 'std', 'min', 'max']])

### 4.7 Lưu dữ liệu đã xử lý (Export Processed Data)

In [ ]:
# ============================================================
# 4.7 — Lưu dữ liệu đã tiền xử lý ra file
# ============================================================

import os

# Thư mục lưu dữ liệu đã xử lý
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)  # Tạo thư mục nếu chưa có

print("="*80)
print("💾 LƯU DỮ LIỆU ĐÃ TIỀN XỬ LÝ")
print("="*80)

# --- 1. Lưu dataset đã clean (trước khi scaling) ---
clean_path = os.path.join(PROCESSED_DIR, 'phishing_email_cleaned.csv')
df_clean.to_csv(clean_path, index=False)
print(f"   ✅ Dataset đã clean       → {clean_path}")

# --- 2. Lưu tập Train đã scaled ---
train_path = os.path.join(PROCESSED_DIR, 'X_train_scaled.csv')
X_train_scaled.to_csv(train_path, index=False)
print(f"   ✅ X_train_scaled         → {train_path}")

# --- 3. Lưu tập Test đã scaled ---
test_path = os.path.join(PROCESSED_DIR, 'X_test_scaled.csv')
X_test_scaled.to_csv(test_path, index=False)
print(f"   ✅ X_test_scaled          → {test_path}")

# --- 4. Lưu nhãn Train và Test ---
y_train.to_csv(os.path.join(PROCESSED_DIR, 'y_train.csv'), index=False, header=['label'])
print(f"   ✅ y_train                → {os.path.join(PROCESSED_DIR, 'y_train.csv')}")

y_test.to_csv(os.path.join(PROCESSED_DIR, 'y_test.csv'), index=False, header=['label'])
print(f"   ✅ y_test                 → {os.path.join(PROCESSED_DIR, 'y_test.csv')}")

print(f"\n🎉 Hoàn tất! Tất cả dữ liệu đã được lưu vào thư mục: {PROCESSED_DIR}")

### 4.8 Tóm tắt toàn bộ quá trình tiền xử lý

In [ ]:
# ============================================================
# 4.8 — Bảng tóm tắt pipeline tiền xử lý
# ============================================================

print("\n" + "="*80)
print("📋 TÓM TẮT TOÀN BỘ PIPELINE TIỀN XỬ LÝ DỮ LIỆU")
print("="*80)

pipeline_summary = [
    ['1. Đọc dữ liệu gốc', f'{df_raw.shape[0]:,} dòng × {df_raw.shape[1]} cột'],
    ['2. Feature Engineering (text → số)', f'Trích xuất {len(numeric_cols)} đặc trưng số từ text'],
    ['3. Xử lý Missing Values', 'Điền chuỗi rỗng (text) / median (số) / drop (target)'],
    ['4. Loại bỏ Duplicates', f'Loại {n_dups:,} dòng trùng lặp'],
    ['5. Xử lý Outliers', f'Cắt ngưỡng IQR × 1.5 (Winsorize) trên {len(numeric_features)} cột'],
    ['6. Label Encoding', f'{label_mapping}'],
    ['7. Train/Test Split', f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,} (80/20, stratified)'],
    ['8. Feature Scaling', 'StandardScaler (fit trên Train → transform Test)'],
    ['9. Dataset sau xử lý', f'{df_clean.shape[0]:,} dòng × {df_clean.shape[1]} cột'],
]

summary_df = pd.DataFrame(pipeline_summary, columns=['Bước', 'Chi tiết'])
display(summary_df)

---

**Nhận xét của Data Scientist:**

> _[Ghi chú phân tích sau khi chạy Section 4 — Ví dụ: Tổng số dòng bị loại bỏ, chiến lược xử lý missing/outliers có hợp lý không, kết quả chuẩn hóa có bình thường không (mean ≈ 0, std ≈ 1), có cần điều chỉnh gì trước khi đưa vào mô hình không, ...]_

---

---

## 📝 Tổng Kết (Final Summary)

### Data Analysis Key Findings

- **Dataset**: ~18,650 email (sau khi loại bỏ index gốc), gồm 2 cột chính: `Email Text` và `Email Type`
- **Class Balance**: Cần kiểm tra tỷ lệ Safe/Phishing sau khi chạy — nếu mất cân bằng > 3:1, cần áp dụng SMOTE hoặc class_weight
- **Feature Engineering**: Trích xuất 10 đặc trưng số từ text (char_count, word_count, uppercase_ratio, url_count, ...)
- **Correlation**: Kiểm tra đa cộng tuyến giữa char_count ↔ word_count (có thể tương quan cao > 0.9)
- **Outliers**: Xử lý bằng Winsorize (cắt ngưỡng IQR) thay vì loại bỏ dòng — giữ nguyên kích thước dataset
- **Data Leakage Prevention**: StandardScaler chỉ fit trên Train, transform trên Test

### Insights & Next Steps

- **Next Step 1**: Tiến hành NLP Preprocessing nâng cao (TF-IDF, Word Embeddings, DistilBERT tokenization) trên cột `Email Text` để tạo feature vectors cho mô hình Deep Learning
- **Next Step 2**: Huấn luyện mô hình phân loại (DistilBERT Fine-tuning) trong kiến trúc Federated Learning

---

> **Notebook được tạo theo chuẩn CRISP-DM**  
> Tác giả: Data Scientist | Dự án: FL Phishing Email Detection